# NOTEBOOK 05 — BUSINESS INSIGHTS & CUSTOMER ACTIVATION

## Objective

Convert the outputs from customer segmentation, churn analysis,
transaction analysis, and recommendation modeling into actionable
business insights.

This notebook focuses on:

1. Customer segment business profiling
2. Churn and retention analysis
3. Revenue-at-risk analysis
4. Customer activation opportunities
5. Bundle and recommendation opportunities
6. Segment-level business actions
7. Executive-level recommendations

The final outputs will provide a clear link between:

Customer Behaviour → Business Risk → Opportunity → Recommended Action

In [1]:
# ================================================================
# 2. IMPORTS
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ================================================================
# 3. PROJECT PATHS
# ================================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
print("Output directory:", OUTPUT_DIR)

Project root: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics
Processed directory: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\data\processed
Output directory: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\outputs


In [3]:
# ================================================================
# 4. AVAILABLE PROCESSED FILES
# ================================================================

print("Available processed files:")
print("-" * 70)

for file in sorted(PROCESSED_DIR.glob("*")):
    if file.is_file():
        print(file.name)

Available processed files:
----------------------------------------------------------------------
active_customer_churn_predictions.csv
churn_model_features.csv
customer_churn_scores.csv
customer_overview.csv
customer_segmentation_features.csv
customer_segments.csv
monthly_sales.csv
orders.csv


In [4]:
# ================================================================
# 4. LOAD PROCESSED DATASETS
# ================================================================

customer_overview = pd.read_csv(
    PROCESSED_DIR / "customer_overview.csv"
)

segmentation_features = pd.read_csv(
    PROCESSED_DIR / "customer_segmentation_features.csv"
)

customer_segments = pd.read_csv(
    PROCESSED_DIR / "customer_segments.csv"
)

customer_churn_scores = pd.read_csv(
    PROCESSED_DIR / "customer_churn_scores.csv"
)

active_customer_churn = pd.read_csv(
    PROCESSED_DIR / "active_customer_churn_predictions.csv"
)

churn_model_features = pd.read_csv(
    PROCESSED_DIR / "churn_model_features.csv"
)

orders = pd.read_csv(
    PROCESSED_DIR / "orders.csv"
)

monthly_sales = pd.read_csv(
    PROCESSED_DIR / "monthly_sales.csv"
)

print("=" * 75)
print("PROCESSED DATASETS LOADED")
print("=" * 75)

print(f"Customer overview:              {customer_overview.shape}")
print(f"Segmentation features:          {segmentation_features.shape}")
print(f"Customer segments:              {customer_segments.shape}")
print(f"Customer churn scores:          {customer_churn_scores.shape}")
print(f"Active customer churn:          {active_customer_churn.shape}")
print(f"Churn model features:            {churn_model_features.shape}")
print(f"Orders:                          {orders.shape}")
print(f"Monthly sales:                   {monthly_sales.shape}")

PROCESSED DATASETS LOADED
Customer overview:              (10965, 20)
Segmentation features:          (10965, 39)
Customer segments:              (10965, 3)
Customer churn scores:          (1450, 5)
Active customer churn:          (7247, 4)
Churn model features:            (7247, 54)
Orders:                          (158115, 9)
Monthly sales:                   (24, 6)


In [5]:
# ================================================================
# 5. DATASET COLUMN INSPECTION
# ================================================================

datasets = {
    "customer_overview": customer_overview,
    "segmentation_features": segmentation_features,
    "customer_segments": customer_segments,
    "customer_churn_scores": customer_churn_scores,
    "active_customer_churn": active_customer_churn,
    "churn_model_features": churn_model_features,
    "orders": orders,
    "monthly_sales": monthly_sales
}

for name, df in datasets.items():

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


CUSTOMER_OVERVIEW
Shape: (10965, 20)
Columns:
['customer_id', 'first_order_date', 'last_order_date', 'frequency', 'monetary', 'average_order_value', 'total_units', 'average_basket_units', 'average_discount', 'signup_date', 'age', 'gender', 'city', 'country', 'household_size', 'acquisition_channel', 'loyalty_tier', 'email_opt_in', 'sms_opt_in', 'app_installed']

SEGMENTATION_FEATURES
Shape: (10965, 39)
Columns:
['customer_id', 'recency', 'frequency', 'monetary', 'average_order_value', 'last_order_date', 'average_basket_size', 'average_discount', 'total_units', 'total_orders', 'weekend_order_share', 'channel_share_call_centre', 'channel_share_in-store', 'channel_share_marketplace', 'channel_share_mobile_app', 'channel_share_web', 'category_share_baby', 'category_share_beverages', 'category_share_fresh_food', 'category_share_grocery', 'category_share_health_&_wellness', 'category_share_home_&_living', 'category_share_household', 'category_share_personal_care', 'category_share_pet_care', 

In [6]:
# ================================================================
# 6. DATASET PREVIEW
# ================================================================

for name, df in datasets.items():

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    display(df.head(3))


CUSTOMER_OVERVIEW


,customer_id,first_order_date,last_order_date,frequency,monetary,average_order_value,total_units,average_basket_units,average_discount,signup_date,age,gender,city,country,household_size,acquisition_channel,loyalty_tier,email_opt_in,sms_opt_in,app_installed
0,CUST100000,2025-08-16 18:59:00,2026-05-22 17:51:00,10,"2,828.2300",282.8230,37,3.7000,6.6667,2025-07-17,28,F,Sharjah,UAE,3,In-Store Signup,Silver,1,1,1
1,CUST100001,2024-10-03 17:09:00,2026-07-26 17:04:00,38,"27,700.3900",728.9576,258,6.7895,2.9704,2024-10-03,35,M,Riyadh,KSA,4,Organic Search,Silver,1,0,1
2,CUST100002,2024-09-14 23:13:00,2024-09-19 14:14:00,2,"1,821.7200",910.8600,24,12.0000,1.0714,2022-03-07,24,M,Abu Dhabi,UAE,8,Organic Search,Gold,1,1,1



SEGMENTATION_FEATURES


,customer_id,recency,frequency,monetary,average_order_value,last_order_date,average_basket_size,average_discount,total_units,total_orders,weekend_order_share,channel_share_call_centre,channel_share_in-store,channel_share_marketplace,channel_share_mobile_app,channel_share_web,category_share_baby,category_share_beverages,category_share_fresh_food,category_share_grocery,category_share_health_&_wellness,category_share_home_&_living,category_share_household,category_share_personal_care,category_share_pet_care,category_share_snacks,private_label_spend_share,signup_date,tenure_months,loyalty_tier,log_recency,log_frequency,log_monetary,log_average_order_value,log_average_basket_size,log_total_units,cluster_5,cluster,segment_name
0,CUST100000,70,10,"2,828.2300",282.8230,2026-05-22 17:51:00,3.7000,6.6667,37,10,0.2000,0.0000,0.1000,0.0000,0.3000,0.6000,0.0162,0.0268,0.0067,0.0169,0.7188,0.0000,0.0022,0.1557,0.0566,0.0000,0.1455,2025-07-17,12.4507,Silver,4.2627,2.3979,7.9478,5.6484,1.5476,3.6376,3,3,Health & Wellness Specialists
1,CUST100001,5,38,"27,700.3900",728.9576,2026-07-26 17:04:00,6.7895,2.9704,258,38,0.4211,0.0000,0.3421,0.0263,0.3684,0.2632,0.0825,0.0132,0.0430,0.0124,0.0347,0.7029,0.0222,0.0373,0.0374,0.0145,0.0415,2024-10-03,21.8791,Silver,1.7918,3.6636,10.2292,6.5930,2.0528,5.5568,1,1,Premium Home & Living Shoppers
2,CUST100002,680,2,"1,821.7200",910.8600,2024-09-19 14:14:00,12.0000,1.0714,24,2,0.5000,0.0000,0.5000,0.5000,0.0000,0.0000,0.0274,0.0000,0.0542,0.0868,0.0000,0.5083,0.0592,0.2426,0.0158,0.0057,0.0951,2022-03-07,52.7924,Gold,6.5236,1.0986,7.5081,6.8155,2.5649,3.2189,1,1,Premium Home & Living Shoppers



CUSTOMER_SEGMENTS


...

,customer_id,churned,last_order_datetime,recency_days,frequency,monetary,average_order_value,orders_last_90d,orders_previous_90d,order_frequency_trend,spend_last_90d,spend_previous_90d,spend_trend,basket_size_last_90d,basket_size_previous_90d,basket_size_trend,web_sessions_last_3m,app_sessions_last_3m,emails_received_last_3m,emails_opened_last_3m,emails_clicked_last_3m,cart_abandonments_last_3m,wishlist_adds_last_3m,web_sessions_previous_3m,app_sessions_previous_3m,emails_received_previous_3m,emails_opened_previous_3m,emails_clicked_previous_3m,web_session_trend,app_session_trend,email_open_trend,email_click_trend,email_open_rate,email_click_rate,ticket_count,mean_csat,mean_first_response_hours,mean_resolution_hours,reopened_ticket_count,average_delivery_delay,returned_order_lines,average_discount_pct,discounted_lines,transaction_lines,discount_dependency,age,household_size,gender,acquisition_channel,loyalty_tier,email_opt_in,sms_opt_in,app_installed,tenure_months
0,CUST100000,0,2026-04-27 18:38:00,3.2236,9,"2,776.1900",308.4656,3.0000,3.0000,0.7500,754.0500,"1,071.1300",0.7033,4.0000,3.0000,1.0000,15.0000,13.0000,9.0000,3.0000,0.0000,4.0000,1.0000,15.0000,18.0000,12.0000,6.0000,1.0000,0.9375,0.6842,0.4286,0.0000,0.3000,0.0000,NaN,NaN,NaN,NaN,NaN,0.0000,4,4.7917,8,24,0.3200,28,3,F,In-Store Signup,Silver,1,1,1,9.4284
1,CUST100001,0,2026-04-19 07:26:00,11.6903,34,"26,980.3000",793.5382,6.0000,5.0000,1.0000,"4,921.3700",908.5700,5.4107,5.3333,6.4000,0.7207,13.0000,38.0000,10.0000,5.0000,1.0000,4.0000,0.0000,20.0000,26.0000,19.0000,8.0000,0.0000,0.6190,1.4074,0.5556,1.0000,0.4545,0.0909,2.0000,4.5000,5.9500,31.0000,1.0000,0.2105,8,2.6316,25,152,0.1634,35,4,M,Organic Search,Silver,1,0,1,18.8568
2,CUST100003,1,2025-11-12 17:13:00,169.2826,3,684.2200,228.0733,0.0000,1.0000,0.0000,0.0000,174.1500,0.0000,0.0000,7.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,6.0000,0.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,0.0000,0,2.9412,4,17,0.2222,45,3,M,Paid Social,Gold,0,1,0,7.4244



ORDERS


,customer_id,order_id,order_datetime,channel,order_value,units,lines,avg_discount,weekday
0,CUST100000,ORD1000000,2025-08-16 18:59:00,Mobile App,529.7000,3,2,5.0000,Saturday
1,CUST100000,ORD1000001,2025-09-05 23:20:00,Web,288.4000,3,3,0.0000,Friday
2,CUST100000,ORD1000002,2025-09-13 20:53:00,Mobile App,132.9100,6,4,3.7500,Saturday



MONTHLY_SALES


,order_datetime,revenue,orders,customers,units,month_number
0,2024-08-01,"1,255,750.6600",4012,2508,23008,8
1,2024-09-01,"1,340,099.0200",4456,2644,25375,9
2,2024-10-01,"1,448,724.1000",4728,2841,26930,10


In [7]:
# ================================================================
# 7. CUSTOMER OVERVIEW
# ================================================================

print("=" * 75)
print("CUSTOMER OVERVIEW")
print("=" * 75)

display(customer_overview.head())

print("\nShape:", customer_overview.shape)

print("\nData types:")
display(customer_overview.dtypes)

print("\nMissing values:")
display(
    customer_overview.isna()
    .sum()
    .sort_values(ascending=False)
)

CUSTOMER OVERVIEW


,customer_id,first_order_date,last_order_date,frequency,monetary,average_order_value,total_units,average_basket_units,average_discount,signup_date,age,gender,city,country,household_size,acquisition_channel,loyalty_tier,email_opt_in,sms_opt_in,app_installed
0,CUST100000,2025-08-16 18:59:00,2026-05-22 17:51:00,10,"2,828.2300",282.8230,37,3.7000,6.6667,2025-07-17,28,F,Sharjah,UAE,3,In-Store Signup,Silver,1,1,1
1,CUST100001,2024-10-03 17:09:00,2026-07-26 17:04:00,38,"27,700.3900",728.9576,258,6.7895,2.9704,2024-10-03,35,M,Riyadh,KSA,4,Organic Search,Silver,1,0,1
2,CUST100002,2024-09-14 23:13:00,2024-09-19 14:14:00,2,"1,821.7200",910.8600,24,12.0000,1.0714,2022-03-07,24,M,Abu Dhabi,UAE,8,Organic Search,Gold,1,1,1
3,CUST100003,2025-09-16 17:18:00,2025-11-12 17:13:00,3,684.2200,228.0733,24,8.0000,2.5595,2025-09-16,45,M,Dubai,UAE,3,Paid Social,Gold,0,1,0
4,CUST100004,2024-08-01 16:17:00,2026-07-16 12:24:00,34,"18,344.3800",539.5406,269,7.9118,1.9233,2022-09-04,45,F,Sharjah,UAE,4,In-Store Signup,Gold,1,0,1



Shape: (10965, 20)

Data types:


customer_id                 str
first_order_date            str
last_order_date             str
frequency                 int64
monetary                float64
average_order_value     float64
total_units               int64
average_basket_units    float64
average_discount        float64
signup_date                 str
age                       int64
gender                      str
city                        str
country                     str
household_size            int64
acquisition_channel         str
loyalty_tier                str
email_opt_in              int64
sms_opt_in                int64
app_installed             int64
dtype: object


Missing values:


customer_id             0
first_order_date        0
last_order_date         0
frequency               0
monetary                0
average_order_value     0
total_units             0
average_basket_units    0
average_discount        0
signup_date             0
age                     0
gender                  0
city                    0
country                 0
household_size          0
acquisition_channel     0
loyalty_tier            0
email_opt_in            0
sms_opt_in              0
app_installed           0
dtype: int64

In [8]:
# ================================================================
# 8. CUSTOMER SEGMENTS
# ================================================================

print("=" * 75)
print("CUSTOMER SEGMENTS")
print("=" * 75)

display(customer_segments.head())

print("\nShape:", customer_segments.shape)

print("\nSegment distribution:")
display(
    customer_segments.iloc[:, -1]
    .value_counts()
)

CUSTOMER SEGMENTS


,customer_id,cluster,segment_name
0,CUST100000,3,Health & Wellness Specialists
1,CUST100001,1,Premium Home & Living Shoppers
2,CUST100002,1,Premium Home & Living Shoppers
3,CUST100003,0,Family & Household Shoppers
4,CUST100004,1,Premium Home & Living Shoppers



Shape: (10965, 3)

Segment distribution:


segment_name
Premium Home & Living Shoppers     3025
Frequent Personal Care Shoppers    2600
Family & Household Shoppers        2389
Health & Wellness Specialists      1631
Lapsed Everyday Shoppers           1320
Name: count, dtype: int64

In [9]:
# ================================================================
# 9. CHURN SCORES
# ================================================================

print("=" * 75)
print("CUSTOMER CHURN SCORES")
print("=" * 75)

display(customer_churn_scores.head())

print("\nShape:", customer_churn_scores.shape)

print("\nColumns:")
print(customer_churn_scores.columns.tolist())

CUSTOMER CHURN SCORES


,customer_id,churned,churn_probability,risk_decile,high_risk
0,CUST104054,1,0.9905,10,1
1,CUST109136,1,0.9880,10,1
2,CUST106355,0,0.9868,10,1
3,CUST104002,0,0.9855,10,1
4,CUST102670,0,0.9854,10,1



Shape: (1450, 5)

Columns:
['customer_id', 'churned', 'churn_probability', 'risk_decile', 'high_risk']


In [10]:
# ================================================================
# 10. ACTIVE CUSTOMER CHURN
# ================================================================

print("=" * 75)
print("ACTIVE CUSTOMER CHURN PREDICTIONS")
print("=" * 75)

display(active_customer_churn.head())

print("\nShape:", active_customer_churn.shape)

print("\nColumns:")
print(active_customer_churn.columns.tolist())

ACTIVE CUSTOMER CHURN PREDICTIONS


,customer_id,churn_probability,risk_decile,high_risk
0,CUST108559,0.9926,10,1
1,CUST106245,0.9921,10,1
2,CUST108698,0.9919,10,1
3,CUST104368,0.9917,10,1
4,CUST103440,0.9913,10,1



Shape: (7247, 4)

Columns:
['customer_id', 'churn_probability', 'risk_decile', 'high_risk']


In [11]:
# ================================================================
# 11. IDENTIFY CUSTOMER ID COLUMNS
# ================================================================

def find_customer_id(df, dataset_name):

    candidates = [
        "customer_id",
        "Customer ID",
        "CustomerID",
        "customerid",
        "customerId"
    ]

    matches = [col for col in candidates if col in df.columns]

    print(f"{dataset_name}: {matches}")

    return matches[0] if matches else None


customer_id_overview = find_customer_id(
    customer_overview,
    "customer_overview"
)

customer_id_segments = find_customer_id(
    customer_segments,
    "customer_segments"
)

customer_id_churn = find_customer_id(
    customer_churn_scores,
    "customer_churn_scores"
)

customer_id_active_churn = find_customer_id(
    active_customer_churn,
    "active_customer_churn"
)

customer_id_features = find_customer_id(
    segmentation_features,
    "segmentation_features"
)

customer_overview: ['customer_id']
customer_segments: ['customer_id']
customer_churn_scores: ['customer_id']
active_customer_churn: ['customer_id']
segmentation_features: ['customer_id']


In [12]:
# ================================================================
# 12. CREATE MASTER CUSTOMER BUSINESS TABLE
# ================================================================

business_customers = segmentation_features.copy()

print("=" * 75)
print("MASTER CUSTOMER BUSINESS TABLE")
print("=" * 75)

print("Rows:", business_customers.shape[0])
print("Columns:", business_customers.shape[1])

print("\nUnique customers:",
      business_customers["customer_id"].nunique())

print("\nDuplicate customer IDs:",
      business_customers["customer_id"].duplicated().sum())

MASTER CUSTOMER BUSINESS TABLE
Rows: 10965
Columns: 39

Unique customers: 10965

Duplicate customer IDs: 0


In [13]:
# ================================================================
# 13. SEGMENT DISTRIBUTION
# ================================================================

segment_distribution = (
    business_customers
    .groupby("segment_name")
    .agg(
        customer_count=("customer_id", "nunique")
    )
    .reset_index()
)

segment_distribution["customer_share_pct"] = (
    segment_distribution["customer_count"]
    / segment_distribution["customer_count"].sum()
    * 100
)

segment_distribution = segment_distribution.sort_values(
    "customer_count",
    ascending=False
)

display(segment_distribution)

,segment_name,customer_count,customer_share_pct
4,Premium Home & Living Shoppers,3025,27.5878
1,Frequent Personal Care Shoppers,2600,23.7118
0,Family & Household Shoppers,2389,21.7875
2,Health & Wellness Specialists,1631,14.8746
3,Lapsed Everyday Shoppers,1320,12.0383


In [14]:
# ================================================================
# 14. SEGMENTATION VALIDATION
# ================================================================

assert business_customers["customer_id"].nunique() == 10965
assert business_customers["segment_name"].notna().all()
assert business_customers["segment_name"].nunique() == 5

print("=" * 75)
print("SEGMENTATION VALIDATION PASSED")
print("=" * 75)

print(f"Customers:        {business_customers['customer_id'].nunique():,}")
print(f"Segments:         {business_customers['segment_name'].nunique()}")
print(f"Missing segments: {business_customers['segment_name'].isna().sum()}")

SEGMENTATION VALIDATION PASSED
Customers:        10,965
Segments:         5
Missing segments: 0


In [15]:
# ================================================================
# 15. SEGMENT BUSINESS PROFILE
# ================================================================

segment_business_profile = (
    business_customers
    .groupby("segment_name")
    .agg(
        customers=("customer_id", "nunique"),
        total_revenue=("monetary", "sum"),
        avg_monetary=("monetary", "mean"),
        total_orders=("total_orders", "sum"),
        avg_frequency=("frequency", "mean"),
        avg_order_value=("average_order_value", "mean"),
        avg_recency_days=("recency", "mean"),
        total_units=("total_units", "sum"),
        avg_basket_size=("average_basket_size", "mean"),
        avg_discount=("average_discount", "mean")
    )
    .reset_index()
)

# Customer share
segment_business_profile["customer_share_pct"] = (
    segment_business_profile["customers"]
    / segment_business_profile["customers"].sum()
    * 100
)

# Revenue share
segment_business_profile["revenue_share_pct"] = (
    segment_business_profile["total_revenue"]
    / segment_business_profile["total_revenue"].sum()
    * 100
)

segment_business_profile = segment_business_profile.sort_values(
    "total_revenue",
    ascending=False
)

display(segment_business_profile)

,segment_name,customers,total_revenue,avg_monetary,total_orders,avg_frequency,avg_order_value,avg_recency_days,total_units,avg_basket_size,avg_discount,customer_share_pct,revenue_share_pct
4,Premium Home & Living Shoppers,3025,"29,876,028.5800","9,876.3731",51732,17.1015,629.8498,102.3147,412068,8.4540,2.7335,27.5878,60.0115
1,Frequent Personal Care Shoppers,2600,"10,396,361.0200","3,998.6004",62744,24.1323,167.8777,96.8723,224907,3.6416,2.6870,23.7118,20.8830
2,Health & Wellness Specialists,1631,"5,299,502.4900","3,249.2351",20251,12.4163,261.1361,155.6603,87287,4.3950,2.7302,14.8746,10.6450
0,Family & Household Shoppers,2389,"3,825,404.7400","1,601.2577",19146,8.0142,273.1481,234.0075,144478,9.1449,4.0716,21.7875,7.6840
3,Lapsed Everyday Shoppers,1320,"386,541.5500",292.8345,4242,3.2136,95.6805,334.8379,20620,5.0465,4.0994,12.0383,0.7764


In [16]:
# ================================================================
# 16. FORMAT SEGMENT PROFILE
# ================================================================

profile_display = segment_business_profile.copy()

profile_display["total_revenue"] = (
    profile_display["total_revenue"].round(2)
)

profile_display["avg_monetary"] = (
    profile_display["avg_monetary"].round(2)
)

profile_display["avg_order_value"] = (
    profile_display["avg_order_value"].round(2)
)

profile_display["avg_recency_days"] = (
    profile_display["avg_recency_days"].round(1)
)

profile_display["avg_frequency"] = (
    profile_display["avg_frequency"].round(2)
)

profile_display["avg_basket_size"] = (
    profile_display["avg_basket_size"].round(2)
)

profile_display["avg_discount"] = (
    profile_display["avg_discount"].round(2)
)

profile_display["customer_share_pct"] = (
    profile_display["customer_share_pct"].round(2)
)

profile_display["revenue_share_pct"] = (
    profile_display["revenue_share_pct"].round(2)
)

display(profile_display)

,segment_name,customers,total_revenue,avg_monetary,total_orders,avg_frequency,avg_order_value,avg_recency_days,total_units,avg_basket_size,avg_discount,customer_share_pct,revenue_share_pct
4,Premium Home & Living Shoppers,3025,"29,876,028.5800","9,876.3700",51732,17.1000,629.8500,102.3000,412068,8.4500,2.7300,27.5900,60.0100
1,Frequent Personal Care Shoppers,2600,"10,396,361.0200","3,998.6000",62744,24.1300,167.8800,96.9000,224907,3.6400,2.6900,23.7100,20.8800
2,Health & Wellness Specialists,1631,"5,299,502.4900","3,249.2400",20251,12.4200,261.1400,155.7000,87287,4.3900,2.7300,14.8700,10.6500
0,Family & Household Shoppers,2389,"3,825,404.7400","1,601.2600",19146,8.0100,273.1500,234.0000,144478,9.1400,4.0700,21.7900,7.6800
3,Lapsed Everyday Shoppers,1320,"386,541.5500",292.8300,4242,3.2100,95.6800,334.8000,20620,5.0500,4.1000,12.0400,0.7800


In [17]:
# ================================================================
# 17. SEGMENT REVENUE CONTRIBUTION
# ================================================================

revenue_contribution = (
    segment_business_profile[
        [
            "segment_name",
            "total_revenue",
            "revenue_share_pct"
        ]
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

display(revenue_contribution)

,segment_name,total_revenue,revenue_share_pct
4,Premium Home & Living Shoppers,"29,876,028.5800",60.0115
1,Frequent Personal Care Shoppers,"10,396,361.0200",20.8830
2,Health & Wellness Specialists,"5,299,502.4900",10.6450
0,Family & Household Shoppers,"3,825,404.7400",7.6840
3,Lapsed Everyday Shoppers,"386,541.5500",0.7764


In [19]:
# ================================================================
# 18. REVENUE PER CUSTOMER
# ================================================================

segment_business_profile["revenue_per_customer"] = (
    segment_business_profile["total_revenue"]
    / segment_business_profile["customers"]
)

segment_business_profile["revenue_per_customer"] = (
    segment_business_profile["revenue_per_customer"].round(2)
)

revenue_per_customer = (
    segment_business_profile[
        [
            "segment_name",
            "customers",
            "customer_share_pct",
            "total_revenue",
            "revenue_share_pct",
            "revenue_per_customer"
        ]
    ]
    .sort_values(
        "revenue_per_customer",
        ascending=False
    )
)

display(revenue_per_customer)

,segment_name,customers,customer_share_pct,total_revenue,revenue_share_pct,revenue_per_customer
4,Premium Home & Living Shoppers,3025,27.5878,"29,876,028.5800",60.0115,"9,876.3700"
1,Frequent Personal Care Shoppers,2600,23.7118,"10,396,361.0200",20.8830,"3,998.6000"
2,Health & Wellness Specialists,1631,14.8746,"5,299,502.4900",10.6450,"3,249.2400"
0,Family & Household Shoppers,2389,21.7875,"3,825,404.7400",7.6840,"1,601.2600"
3,Lapsed Everyday Shoppers,1320,12.0383,"386,541.5500",0.7764,292.8300


In [20]:
# ================================================================
# 19. MERGE ACTIVE CUSTOMER CHURN PREDICTIONS
# ================================================================

active_churn_merge = active_customer_churn[
    [
        "customer_id",
        "churn_probability",
        "risk_decile",
        "high_risk"
    ]
].copy()

business_with_churn = business_customers.merge(
    active_churn_merge,
    on="customer_id",
    how="left"
)

print("=" * 75)
print("ACTIVE CHURN MERGE")
print("=" * 75)

print("Original customers:",
      len(business_customers))

print("After merge:",
      len(business_with_churn))

print("Customers with churn prediction:",
      business_with_churn["churn_probability"].notna().sum())

ACTIVE CHURN MERGE
Original customers: 10965
After merge: 10965
Customers with churn prediction: 7247


In [21]:
# ================================================================
# 20. CHURN COVERAGE VALIDATION
# ================================================================

churn_coverage = (
    business_with_churn["churn_probability"]
    .notna()
    .sum()
)

print("=" * 75)
print("CHURN COVERAGE")
print("=" * 75)

print(f"Segmented customers:       {len(business_customers):,}")
print(f"Active churn predictions:  {churn_coverage:,}")
print(
    f"Coverage:                  "
    f"{churn_coverage / len(business_customers) * 100:.2f}%"
)

CHURN COVERAGE
Segmented customers:       10,965
Active churn predictions:  7,247
Coverage:                  66.09%


In [22]:
# ================================================================
# 21. CHURN RISK BY SEGMENT
# ================================================================

churn_by_segment = (
    business_with_churn[
        business_with_churn["churn_probability"].notna()
    ]
    .groupby("segment_name")
    .agg(
        customers=("customer_id", "nunique"),
        avg_churn_probability=("churn_probability", "mean"),
        high_risk_customers=("high_risk", "sum")
    )
    .reset_index()
)

churn_by_segment["high_risk_rate_pct"] = (
    churn_by_segment["high_risk_customers"]
    / churn_by_segment["customers"]
    * 100
)

churn_by_segment = churn_by_segment.sort_values(
    "avg_churn_probability",
    ascending=False
)

display(churn_by_segment)

,segment_name,customers,avg_churn_probability,high_risk_customers,high_risk_rate_pct
3,Lapsed Everyday Shoppers,311,0.4906,101.0000,32.4759
0,Family & Household Shoppers,1231,0.3825,241.0000,19.5776
2,Health & Wellness Specialists,1098,0.2489,122.0000,11.1111
4,Premium Home & Living Shoppers,2484,0.1795,171.0000,6.8841
1,Frequent Personal Care Shoppers,2123,0.1513,161.0000,7.5836


In [24]:
# ================================================================
# 22. REVENUE AT RISK
# ================================================================

business_with_churn["revenue_at_risk"] = (
    business_with_churn["monetary"]
    * business_with_churn["churn_probability"]
)

revenue_risk_by_segment = (
    business_with_churn[
        business_with_churn["churn_probability"].notna()
    ]
    .groupby("segment_name")
    .agg(
        customers=("customer_id", "nunique"),
        current_revenue=("monetary", "sum"),
        revenue_at_risk=("revenue_at_risk", "sum"),
        high_risk_customers=("high_risk", "sum")
    )
    .reset_index()
)

revenue_risk_by_segment["risk_pct_of_revenue"] = (
    revenue_risk_by_segment["revenue_at_risk"]
    / revenue_risk_by_segment["current_revenue"]
    * 100
)

revenue_risk_by_segment = revenue_risk_by_segment.sort_values(
    "revenue_at_risk",
    ascending=False
)

display(revenue_risk_by_segment)

,segment_name,customers,current_revenue,revenue_at_risk,high_risk_customers,risk_pct_of_revenue
4,Premium Home & Living Shoppers,2484,"27,543,820.2200","3,395,428.4446",171.0000,12.3274
1,Frequent Personal Care Shoppers,2123,"9,636,100.9300","1,033,208.7254",161.0000,10.7223
0,Family & Household Shoppers,1231,"2,624,145.8500","928,580.0808",241.0000,35.3860
2,Health & Wellness Specialists,1098,"4,582,911.5700","760,507.3819",122.0000,16.5944
3,Lapsed Everyday Shoppers,311,"143,317.5400","63,431.5329",101.0000,44.2594


In [25]:
# ================================================================
# 23. PRIORITY SEGMENT ANALYSIS
# ================================================================

priority_segments = (
    segment_business_profile[
        [
            "segment_name",
            "customers",
            "customer_share_pct",
            "total_revenue",
            "revenue_share_pct",
            "revenue_per_customer"
        ]
    ]
    .merge(
        churn_by_segment[
            [
                "segment_name",
                "avg_churn_probability",
                "high_risk_customers",
                "high_risk_rate_pct"
            ]
        ],
        on="segment_name",
        how="left"
    )
    .merge(
        revenue_risk_by_segment[
            [
                "segment_name",
                "revenue_at_risk",
                "risk_pct_of_revenue"
            ]
        ],
        on="segment_name",
        how="left"
    )
)

display(
    priority_segments.sort_values(
        "revenue_at_risk",
        ascending=False
    )
)

,segment_name,customers,customer_share_pct,total_revenue,revenue_share_pct,revenue_per_customer,avg_churn_probability,high_risk_customers,high_risk_rate_pct,revenue_at_risk,risk_pct_of_revenue
0,Premium Home & Living Shoppers,3025,27.5878,"29,876,028.5800",60.0115,"9,876.3700",0.1795,171.0000,6.8841,"3,395,428.4446",12.3274
1,Frequent Personal Care Shoppers,2600,23.7118,"10,396,361.0200",20.8830,"3,998.6000",0.1513,161.0000,7.5836,"1,033,208.7254",10.7223
3,Family & Household Shoppers,2389,21.7875,"3,825,404.7400",7.6840,"1,601.2600",0.3825,241.0000,19.5776,"928,580.0808",35.3860
2,Health & Wellness Specialists,1631,14.8746,"5,299,502.4900",10.6450,"3,249.2400",0.2489,122.0000,11.1111,"760,507.3819",16.5944
4,Lapsed Everyday Shoppers,1320,12.0383,"386,541.5500",0.7764,292.8300,0.4906,101.0000,32.4759,"63,431.5329",44.2594


In [26]:
# ================================================================
# 24. SAVE SEGMENT BUSINESS PROFILE
# ================================================================

segment_profile_path = (
    OUTPUT_DIR / "segment_business_profile.csv"
)

segment_business_profile.to_csv(
    segment_profile_path,
    index=False
)

print("Saved:")
print(segment_profile_path)

Saved:
C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\outputs\segment_business_profile.csv


In [30]:
# ================================================================
# LOAD NOTEBOOK 04 — MARKET BASKET OUTPUTS
# ================================================================

MARKET_BASKET_DIR = OUTPUT_DIR / "tables" / "market_basket"

print("=" * 75)
print("NOTEBOOK 04 OUTPUT DIRECTORY")
print("=" * 75)

print(MARKET_BASKET_DIR)

print("\nAvailable files:")
for file in sorted(MARKET_BASKET_DIR.glob("*.csv")):
    print("-", file.name)

NOTEBOOK 04 OUTPUT DIRECTORY
C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\outputs\tables\market_basket

Available files:
- bundle_opportunities.csv
- category_association_rules.csv
- item_item_similarity.csv
- recommendation_model_comparison.csv
- segment_association_rules.csv
- sku_association_rules.csv
- subcategory_association_rules.csv


In [31]:
# ================================================================
# LOAD NOTEBOOK 04 OUTPUT FILES
# ================================================================

bundle_opportunities = pd.read_csv(
    MARKET_BASKET_DIR / "bundle_opportunities.csv"
)

sku_rules = pd.read_csv(
    MARKET_BASKET_DIR / "sku_association_rules.csv"
)

category_rules = pd.read_csv(
    MARKET_BASKET_DIR / "category_association_rules.csv"
)

subcategory_rules = pd.read_csv(
    MARKET_BASKET_DIR / "subcategory_association_rules.csv"
)

segment_rules = pd.read_csv(
    MARKET_BASKET_DIR / "segment_association_rules.csv"
)

recommendation_comparison = pd.read_csv(
    MARKET_BASKET_DIR / "recommendation_model_comparison.csv"
)

item_item_similarity = pd.read_csv(
    MARKET_BASKET_DIR / "item_item_similarity.csv"
)

print("=" * 75)
print("NOTEBOOK 04 OUTPUTS LOADED")
print("=" * 75)

print(f"Bundle opportunities:       {len(bundle_opportunities):,}")
print(f"SKU association rules:      {len(sku_rules):,}")
print(f"Category rules:             {len(category_rules):,}")
print(f"Subcategory rules:          {len(subcategory_rules):,}")
print(f"Segment rules:              {len(segment_rules):,}")
print(f"Recommendation models:      {len(recommendation_comparison):,}")
print(f"Item-item similarities:     {len(item_item_similarity):,}")

NOTEBOOK 04 OUTPUTS LOADED
Bundle opportunities:       50
SKU association rules:      116
Category rules:             1,984
Subcategory rules:          388
Segment rules:              5
Recommendation models:      3
Item-item similarities:     3,910


In [32]:
# ================================================================
# NOTEBOOK 05 VALIDATION — PART 1
# ================================================================

print("=" * 75)
print("NOTEBOOK 05 — BUSINESS PROFILE VALIDATION")
print("=" * 75)

print(
    f"Segmented customers:       "
    f"{business_customers['customer_id'].nunique():,}"
)

print(
    f"Segments:                   "
    f"{business_customers['segment_name'].nunique()}"
)

print(
    f"Active churn predictions:   "
    f"{active_customer_churn['customer_id'].nunique():,}"
)

print(
    f"High-risk active customers: "
    f"{active_customer_churn['high_risk'].sum():,}"
)

print(
    f"Bundle opportunities:       "
    f"{len(bundle_opportunities):,}"
)

print(
    f"Revenue at risk:            "
    f"AED {revenue_risk_by_segment['revenue_at_risk'].sum():,.2f}"
)

print("\n" + "=" * 75)
print("SEGMENT PROFILE")
print("=" * 75)

display(
    priority_segments.sort_values(
        "revenue_at_risk",
        ascending=False
    )
)

NOTEBOOK 05 — BUSINESS PROFILE VALIDATION
Segmented customers:       10,965
Segments:                   5
Active churn predictions:   7,247
High-risk active customers: 796
Bundle opportunities:       50
Revenue at risk:            AED 6,181,156.17

SEGMENT PROFILE


,segment_name,customers,customer_share_pct,total_revenue,revenue_share_pct,revenue_per_customer,avg_churn_probability,high_risk_customers,high_risk_rate_pct,revenue_at_risk,risk_pct_of_revenue
0,Premium Home & Living Shoppers,3025,27.5878,"29,876,028.5800",60.0115,"9,876.3700",0.1795,171.0000,6.8841,"3,395,428.4446",12.3274
1,Frequent Personal Care Shoppers,2600,23.7118,"10,396,361.0200",20.8830,"3,998.6000",0.1513,161.0000,7.5836,"1,033,208.7254",10.7223
3,Family & Household Shoppers,2389,21.7875,"3,825,404.7400",7.6840,"1,601.2600",0.3825,241.0000,19.5776,"928,580.0808",35.3860
2,Health & Wellness Specialists,1631,14.8746,"5,299,502.4900",10.6450,"3,249.2400",0.2489,122.0000,11.1111,"760,507.3819",16.5944
4,Lapsed Everyday Shoppers,1320,12.0383,"386,541.5500",0.7764,292.8300,0.4906,101.0000,32.4759,"63,431.5329",44.2594


In [35]:
[v for v in dir() if any(x in v.lower() for x in ["churn", "pred", "risk"])]

['active_churn_merge',
 'active_customer_churn',
 'business_with_churn',
 'churn_by_segment',
 'churn_coverage',
 'churn_model_features',
 'customer_churn_scores',
 'customer_id_active_churn',
 'customer_id_churn',
 'revenue_risk_by_segment']

In [36]:
print("active_customer_churn")
print(active_customer_churn.shape)
print(active_customer_churn.columns.tolist())
print(active_customer_churn.head())

print("\n" + "="*75 + "\n")

print("customer_churn_scores")
print(customer_churn_scores.shape)
print(customer_churn_scores.columns.tolist())
print(customer_churn_scores.head())

active_customer_churn
(7247, 4)
['customer_id', 'churn_probability', 'risk_decile', 'high_risk']
  customer_id  churn_probability  risk_decile  high_risk
0  CUST108559             0.9926           10          1
1  CUST106245             0.9921           10          1
2  CUST108698             0.9919           10          1
3  CUST104368             0.9917           10          1
4  CUST103440             0.9913           10          1


customer_churn_scores
(1450, 5)
['customer_id', 'churned', 'churn_probability', 'risk_decile', 'high_risk']
  customer_id  churned  churn_probability  risk_decile  high_risk
0  CUST104054        1             0.9905           10          1
1  CUST109136        1             0.9880           10          1
2  CUST106355        0             0.9868           10          1
3  CUST104002        0             0.9855           10          1
4  CUST102670        0             0.9854           10          1


In [37]:
print("active_churn_merge")
print(active_churn_merge.shape)
print(active_churn_merge.columns.tolist())
print(active_churn_merge.head())

active_churn_merge
(7247, 4)
['customer_id', 'churn_probability', 'risk_decile', 'high_risk']
  customer_id  churn_probability  risk_decile  high_risk
0  CUST108559             0.9926           10          1
1  CUST106245             0.9921           10          1
2  CUST108698             0.9919           10          1
3  CUST104368             0.9917           10          1
4  CUST103440             0.9913           10          1


In [38]:
# ================================================================
# NOTEBOOK 05 — SIMPLE VALIDATION
# ================================================================

print("=" * 75)
print("NOTEBOOK 05 — VALIDATION")
print("=" * 75)

# 1. Active churn predictions
print("\n1. ACTIVE CHURN")
print("Rows:", len(active_customer_churn))
print("Unique customers:", active_customer_churn["customer_id"].nunique())
print("Expected: 7,247")

# 2. High-risk customers
print("\n2. HIGH-RISK CUSTOMERS")
print("High-risk:", active_customer_churn["high_risk"].sum())
print("Expected: 796")

# 3. Risk deciles
print("\n3. RISK DECILES")
print(active_customer_churn["risk_decile"].value_counts().sort_index())

# 4. Revenue at risk
print("\n4. REVENUE AT RISK")
print(revenue_risk_by_segment)

print(
    "\nTotal revenue at risk:",
    revenue_risk_by_segment["revenue_at_risk"].sum()
)
print("Expected: AED 6,181,156.17")

# 5. Segment churn validation
print("\n5. CHURN BY SEGMENT")
print(churn_by_segment)

print(
    "\nTotal high-risk by segment:",
    churn_by_segment["high_risk_customers"].sum()
)
print("Expected: 796")

# 6. Bundle opportunities
print("\n6. BUNDLE OPPORTUNITIES")
print("Bundle opportunities:", len(bundle_opportunities))
print("Expected: 50")

print("\n" + "=" * 75)
print("VALIDATION COMPLETE")
print("=" * 75)

NOTEBOOK 05 — VALIDATION

1. ACTIVE CHURN
Rows: 7247
Unique customers: 7247
Expected: 7,247

2. HIGH-RISK CUSTOMERS
High-risk: 796
Expected: 796

3. RISK DECILES
risk_decile
1     725
2     725
3     724
4     725
5     725
6     724
7     725
8     724
9     725
10    725
Name: count, dtype: int64

4. REVENUE AT RISK
                      segment_name  customers  current_revenue  \
4   Premium Home & Living Shoppers       2484  27,543,820.2200   
1  Frequent Personal Care Shoppers       2123   9,636,100.9300   
0      Family & Household Shoppers       1231   2,624,145.8500   
2    Health & Wellness Specialists       1098   4,582,911.5700   
3         Lapsed Everyday Shoppers        311     143,317.5400   

   revenue_at_risk  high_risk_customers  risk_pct_of_revenue  
4   3,395,428.4446             171.0000              12.3274  
1   1,033,208.7254             161.0000              10.7223  
0     928,580.0808             241.0000              35.3860  
2     760,507.3819             

In [39]:
print(active_customer_churn.columns.tolist())
print(churn_by_segment.columns.tolist())
print(revenue_risk_by_segment.columns.tolist())
print(bundle_opportunities.columns.tolist())

['customer_id', 'churn_probability', 'risk_decile', 'high_risk']
['segment_name', 'customers', 'avg_churn_probability', 'high_risk_customers', 'high_risk_rate_pct']
['segment_name', 'customers', 'current_revenue', 'revenue_at_risk', 'high_risk_customers', 'risk_pct_of_revenue']
['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski', 'antecedents_names', 'consequents_names', 'antecedent_id', 'consequent_id', 'product_id_x', 'antecedent_name', 'brand_x', 'antecedent_category', 'antecedent_subcategory', 'unit_price_aed_x', 'product_id_y', 'consequent_name', 'brand_y', 'consequent_category', 'consequent_subcategory', 'unit_price_aed_y', 'orders_with_rule', 'antecedent_orders', 'estimated_target_orders', 'consequent_price_aed', 'estimated_revenue_opportunity_aed']


In [40]:
print(business_with_churn.shape)
print(business_with_churn.columns.tolist())
print(business_with_churn.head())

(10965, 43)
['customer_id', 'recency', 'frequency', 'monetary', 'average_order_value', 'last_order_date', 'average_basket_size', 'average_discount', 'total_units', 'total_orders', 'weekend_order_share', 'channel_share_call_centre', 'channel_share_in-store', 'channel_share_marketplace', 'channel_share_mobile_app', 'channel_share_web', 'category_share_baby', 'category_share_beverages', 'category_share_fresh_food', 'category_share_grocery', 'category_share_health_&_wellness', 'category_share_home_&_living', 'category_share_household', 'category_share_personal_care', 'category_share_pet_care', 'category_share_snacks', 'private_label_spend_share', 'signup_date', 'tenure_months', 'loyalty_tier', 'log_recency', 'log_frequency', 'log_monetary', 'log_average_order_value', 'log_average_basket_size', 'log_total_units', 'cluster_5', 'cluster', 'segment_name', 'churn_probability', 'risk_decile', 'high_risk', 'revenue_at_risk']
  customer_id  recency  frequency    monetary  average_order_value  \
0 